In [0]:
%sql
SELECT *
FROM bronze.medicamento
LIMIT 20;

In [0]:
%sql
SELECT
    id_medicamento,
    nombre,
    tipo,
    fabricante,
    stock
FROM bronze.medicamento;

In [0]:
import requests

url = "https://api.fda.gov/drug/label.json"

params = {
    "search": 'openfda.generic_name:"omeprazole"',
    "limit": 3
}

respuesta = requests.get(url, params=params)

print("Código:", respuesta.status_code)
print(respuesta.text[:3000])

In [0]:
%sql
SELECT
    id_medicamento,
    nombre,
    tipo,
    fabricante,
    stock
FROM bronze.medicamento
ORDER BY id_medicamento;

In [0]:
import requests
import json
from pyspark.sql import Row

In [0]:
homologacion = {
    1: "acetaminophen",
    2: "ibuprofen",
    3: "amoxicillin",
    4: "omeprazole",
    5: "losartan",
    6: "metformin",
    7: "albuterol",
    8: "atorvastatin",
    9: "amlodipine",
    10: "azithromycin",
    11: "loratadine",
    12: "diclofenac",
    13: "enalapril",
    14: "acetaminophen",
    15: "omeprazole",
    16: "ceftriaxone",
    17: "insulin human",
    18: "prednisone",
    19: "budesonide",
    20: "amoxicillin and clavulanate potassium"
}

In [0]:
medicamentos = spark.sql("""
    SELECT
        id_medicamento,
        nombre,
        tipo,
        fabricante,
        stock
    FROM bronze.medicamento
    ORDER BY id_medicamento
""")

medicamentos.show(20, False)

In [0]:
url = "https://api.fda.gov/drug/label.json"

resultados = []

for medicamento in medicamentos.collect():

    id_medicamento = medicamento["id_medicamento"]
    nombre = medicamento["nombre"]

    nombre_api = homologacion.get(id_medicamento)

    params = {
        "search": 'openfda.generic_name:"' + nombre_api + '"',
        "limit": 1
    }

    respuesta = requests.get(url, params=params)

    if respuesta.status_code == 200:

        datos = respuesta.json()

        if "results" in datos and len(datos["results"]) > 0:

            resultado_api = datos["results"][0]

            resultados.append(
                Row(
                    id_medicamento=id_medicamento,
                    nombre_medicamento=nombre,
                    nombre_api=nombre_api,
                    codigo_respuesta=respuesta.status_code,
                    encontrado=True,
                    raw_json=json.dumps(resultado_api)
                )
            )

        else:

            resultados.append(
                Row(
                    id_medicamento=id_medicamento,
                    nombre_medicamento=nombre,
                    nombre_api=nombre_api,
                    codigo_respuesta=respuesta.status_code,
                    encontrado=False,
                    raw_json=None
                )
            )

    else:

        resultados.append(
            Row(
                id_medicamento=id_medicamento,
                nombre_medicamento=nombre,
                nombre_api=nombre_api,
                codigo_respuesta=respuesta.status_code,
                encontrado=False,
                raw_json=None
            )
        )

print("Medicamentos procesados:", len(resultados))

In [0]:
df_openfda = spark.createDataFrame(resultados)

df_openfda.write \
    .mode("overwrite") \
    .saveAsTable("bronze.openfda_medicamentos")

In [0]:
%sql
SELECT *
FROM bronze.openfda_medicamentos;

In [0]:
%sql
SELECT
    id_medicamento,
    nombre_medicamento,
    nombre_api,
    codigo_respuesta,
    encontrado
FROM bronze.openfda_medicamentos
ORDER BY id_medicamento;

In [0]:
%sql
SELECT
    encontrado,
    COUNT(*) AS cantidad
FROM bronze.openfda_medicamentos
GROUP BY encontrado;

In [0]:
%sql
SELECT COUNT(*) AS total
FROM bronze.openfda_medicamentos;

In [0]:
%sql
SELECT
    id_medicamento,
    nombre_medicamento,
    nombre_api,
    raw_json
FROM bronze.openfda_medicamentos
WHERE encontrado = true
LIMIT 2;

In [0]:
%sql
DESCRIBE bronze.receta;

In [0]:
%sql
DESCRIBE bronze.detalle_receta;

In [0]:
%sql
SELECT *
FROM bronze.receta
LIMIT 5;

In [0]:
%sql
SELECT *
FROM bronze.detalle_receta
LIMIT 5;

In [0]:
print ("Actualizado")